# QX02 — Porte quantistiche e algoritmi fondamentali con Qiskit

**Obiettivo:** visualizzare e comprendere, passo dopo passo, le porte elementari e i protocolli/algoritmi
che costituiscono il nucleo del calcolo quantistico.

**Contenuti:**
1. Porte quantistiche (introduzione)
2. Porte di Pauli (I, X, Y, Z)
3. Porta di Hadamard
4. Porta CNOT (controlled-NOT)
5. Porta Z (fase)
6. Misure nel circuito
7. Superdense coding (Alice e Bob)
8. Teletrasporto quantistico
9. Algoritmo di Bernstein–Vazirani
10. Algoritmo di Deutsch
11. Algoritmo di Grover
12. Algoritmo di Shor (demo educativa)

**Prerequisiti:** ambiente `.venv-quantum` attivo con `pip install -r quantum/requirements.txt`.

> Per eseguire su **hardware IBM** (opzionale) copia `qiskit_secrets.example.py` in `qiskit_secrets.local.py`
> e inserisci il token. Il file locale è nel `.gitignore` e non verrà committato.

## 0. Setup e funzioni di utilità

Un **qubit** è l'unità fondamentale di informazione quantistica. Il suo stato generale è una
**sovrapposizione** nella base computazionale:

$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle, \qquad |\alpha|^2 + |\beta|^2 = 1$$

Le **porte quantistiche** sono operazioni **unitarie** $U$ tali che $U^\dagger U = I$.
Applicano una rotazione/evoluzione reversibile dello stato, a differenza della **misura** che è
stocastica e collassa la sovrapposizione.

In [ ]:
import sys
from pathlib import Path

# Su Windows il terminale spesso usa cp1252: forziamo UTF-8 per i diagrammi Qiskit
try:
    sys.stdout.reconfigure(encoding="utf-8")
except (AttributeError, ValueError):
    pass

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from IPython.display import display, Markdown

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector, Operator, Pauli
from qiskit.visualization import plot_histogram, plot_bloch_multivector

SIM = AerSimulator()


def run_circuit(qc: QuantumCircuit, shots: int = 1024, show_hist: bool = True):
    """Esegue il circuito sul simulatore Aer e restituisce i conteggi delle misure."""
    tqc = transpile(qc, SIM)
    result = SIM.run(tqc, shots=shots).result()
    counts = result.get_counts()
    if show_hist and qc.num_clbits > 0:
        display(plot_histogram(counts))
    return counts


def show_bloch(qc: QuantumCircuit, title: str = ""):
    """Visualizza lo stato finale (prima delle misure) sulla sfera di Bloch."""
    qc_pure = qc.remove_final_measurements(inplace=False)
    state = Statevector.from_instruction(qc_pure)
    fig = plot_bloch_multivector(state)
    if title:
        fig.suptitle(title)
    plt.show()
    return state


def draw_mpl(qc: QuantumCircuit, title: str = ""):
    """Disegna il circuito con matplotlib."""
    fig = qc.draw(output="mpl", fold=-1)
    if title:
        fig.suptitle(title)
    return fig


def try_ibm_backend():
    """Carica un backend IBM se esiste qiskit_secrets.local.py (opzionale)."""
    secrets_file = Path("qiskit_secrets.local.py")
    if not secrets_file.exists():
        print("Nessun token IBM: uso solo AerSimulator (locale).")
        return None
    try:
        import importlib.util
        spec = importlib.util.spec_from_file_location("qiskit_secrets", secrets_file)
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        if not getattr(mod, "IBM_TOKEN", ""):
            print("Token IBM vuoto in qiskit_secrets.local.py — uso Aer.")
            return None
        from qiskit_ibm_runtime import QiskitRuntimeService
        kwargs = {"channel": getattr(mod, "IBM_CHANNEL", "ibm_quantum"), "token": mod.IBM_TOKEN}
        if getattr(mod, "IBM_INSTANCE", ""):
            kwargs["instance"] = mod.IBM_INSTANCE
        service = QiskitRuntimeService(**kwargs)
        backend = service.least_busy(operational=True, simulator=False)
        print(f"Backend IBM selezionato: {backend.name}")
        return backend
    except ImportError:
        print("Per hardware IBM installa: pip install qiskit-ibm-runtime")
        return None
    except Exception as exc:
        print(f"Errore connessione IBM: {exc}")
        return None

print("Setup completato. Simulatore:", SIM)

---
## 1. Porte quantistiche — panoramica

Ogni porta a 1 qubit è una matrice unitaria $2\times 2$. Le combinano in sequenza per costruire
circuiti; su più qubit usiamo il **prodotto tensoriale** (es. $H \otimes I$ applica Hadamard solo
al primo qubit).

| Porta | Simbolo | Effetto su $|0\rangle$ | Ruolo fisico |
|-------|---------|----------------------|--------------|
| Identità | $I$ | $|0\rangle$ | Nessuna operazione |
| Pauli-X | $X$ | $|1\rangle$ | NOT quantistico (bit-flip) |
| Pauli-Y | $Y$ | $i|1\rangle$ | Bit-flip + phase-flip |
| Pauli-Z | $Z$ | $|0\rangle$ | Phase-flip: $|1\rangle \mapsto -|1\rangle$ |
| Hadamard | $H$ | $\frac{|0\rangle+|1\rangle}{\sqrt{2}}$ | Crea sovrapposizione |

Verifichiamo l'unitarietà con `Operator`: $U^\dagger U = I$.

In [ ]:
gates_1q = {
    "I": Operator(np.eye(2)),
    "X": Operator(Pauli("X")),
    "Y": Operator(Pauli("Y")),
    "Z": Operator(Pauli("Z")),
    "H": Operator([[1, 1], [1, -1]] / np.sqrt(2)),
}

for name, op in gates_1q.items():
    U = op.data
    unitary = np.allclose(U.conj().T @ U, np.eye(2))
    print(f"{name}: unitaria = {unitary}")

---
## 2. Porte di Pauli (I, X, Y, Z)

Le **matrici di Pauli** formano una base per le operazioni a 1 qubit (con $I$):

$$X = \begin{pmatrix}0&1\\1&0\end{pmatrix},\quad
Y = \begin{pmatrix}0&-i\\i&0\end{pmatrix},\quad
Z = \begin{pmatrix}1&0\\0&-1\end{pmatrix}$$

**Proprietà importanti:**
- $X$, $Y$, $Z$ si **anticommutano**: $XY = -YX$, ecc.
- Sono **autoinversi**: $X^2 = Y^2 = Z^2 = I$
- $X$ scambia $|0\rangle \leftrightarrow |1\rangle$ (come il NOT classico su base computazionale)
- $Z$ aggiunge una fase $-1$ solo allo stato $|1\rangle$

Sulla **sfera di Bloch**, $X$, $Y$, $Z$ corrispondono a rotazioni di $\pi$ attorno agli assi $x$, $y$, $z$.

In [ ]:
for gate in ["x", "y", "z"]:
    qc = QuantumCircuit(1)
    getattr(qc, gate)(0)
    draw_mpl(qc, title=f"Porta Pauli-{gate.upper()} su |0⟩")
    plt.show()
    show_bloch(qc, title=f"Stato dopo {gate.upper()}|0⟩")

---
## 3. Porta di Hadamard

La porta **Hadamard** crea una **sovrapposizione uniforme**:

$$H = \frac{1}{\sqrt{2}}\begin{pmatrix}1&1\\1&-1\end{pmatrix},\qquad
H|0\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}},\quad
H|1\rangle = \frac{|0\rangle - |1\rangle}{\sqrt{2}}$$

È la sua stessa inversa: $H^2 = I$. Geometricamente, è una rotazione di $\pi$ attorno all'asse
$(\hat{x}+\hat{z})/\sqrt{2}$ sulla sfera di Bloch.

Hadamard è fondamentale per: superposizione, stati Bell, algoritmi (Grover, Bernstein–Vazirani, ecc.).

In [ ]:
qc_h0 = QuantumCircuit(1, 1)
qc_h0.h(0)
qc_h0.measure(0, 0)

draw_mpl(qc_h0, title="H|0⟩ — superposizione |+⟩")
plt.show()
show_bloch(QuantumCircuit(1).h(0), title="Bloch: |+⟩ = H|0⟩")

counts = run_circuit(qc_h0, shots=2048)
print("Distribuzione attesa: ~50% |0⟩, ~50% |1⟩")
print(counts)

---
## 4. Porta CNOT (Controlled-NOT)

La **CNOT** è una porta a **2 qubit**: il qubit di controllo (●) determina se applicare $X$ sul target (⊕).

$$\mathrm{CNOT} = \begin{pmatrix}
1&0&0&0\\0&1&0&0\\0&0&0&1\\0&0&1&0
\end{pmatrix}$$

**Tabella verità (base computazionale):**
$$|00\rangle\mapsto|00\rangle,\; |01\rangle\mapsto|01\rangle,\; |10\rangle\mapsto|11\rangle,\; |11\rangle\mapsto|10\rangle$$

Combinata con $H$, genera **entanglement**. Il circuito $H$ sul controllo seguito da CNOT produce lo
stato Bell $|\Phi^+\rangle = (|00\rangle+|11\rangle)/\sqrt{2}$.

In [ ]:
qc_bell = QuantumCircuit(2, 2)
qc_bell.h(0)
qc_bell.cx(0, 1)
qc_bell.measure([0, 1], [0, 1])

draw_mpl(qc_bell, title="Stato Bell |Φ⁺⟩ = CNOT · (H⊗I)|00⟩")
plt.show()

counts = run_circuit(qc_bell, shots=2048)
print("Solo esiti 00 e 11 — correlazione perfetta (entanglement)")
print(counts)

---
## 5. Porta Z (fase)

La porta **Z** è una **rotazione di fase** di $\pi$ attorno all'asse $z$:

$$Z|0\rangle = |0\rangle,\qquad Z|1\rangle = -|1\rangle$$

Su una sovrapposizione $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$:
$$Z|\psi\rangle = \alpha|0\rangle - \beta|1\rangle$$

La fase relativa tra $|0\rangle$ e $|1\rangle$ cambia segno, ma le **probabilità** $|\alpha|^2$ e $|\beta|^2$
restano identiche — la fase non è osservabile con una singola misura nella base computazionale.
Per vederne l'effetto serve **interferenza** (es. applicare $H$ dopo $Z$).

In [ ]:
qc_z_phase = QuantumCircuit(1, 1)
qc_z_phase.h(0)   # |+⟩
qc_z_phase.z(0)   # |−⟩
qc_z_phase.h(0)   # torna a |1⟩
qc_z_phase.measure(0, 0)

draw_mpl(qc_z_phase, title="H·Z·H|0⟩ = |1⟩ — la fase diventa osservabile")
plt.show()

counts = run_circuit(qc_z_phase, shots=1024)
print("Risultato: quasi sempre |1⟩ (interferenza distruttiva su |0⟩)")
print(counts)

---
## 6. Misure nel circuito

La **misura** proietta lo stato sul autospazio dell'osservabile misurato. Nella base computazionale:

$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle \xrightarrow{\text{misura}} \begin{cases}|0\rangle & P = |\alpha|^2 \\ |1\rangle & P = |\beta|^2\end{cases}$$

In Qiskit:
- `qc.measure(qubit, classical_bit)` collega un qubit a un **bit classico**
- `measure_all()` misura tutti i qubit nell'ordine $q_{n-1}\ldots q_0$
- Dopo la misura il qubit **collassa**: non si possono applicare porte unitarie sullo stesso qubit
  nello stesso "shot" (nel simulatore si modella come proiezione + reset implicito)

**Regola di Born:** la probabilità dell'esito $k$ è $|\langle k|\psi\rangle|^2$.

In [ ]:
qc_meas = QuantumCircuit(2, 2)
qc_meas.h(0)
qc_meas.cx(0, 1)
qc_meas.barrier(label="prima della misura")
qc_meas.measure(0, 0)  # misura q0 -> c0
qc_meas.measure(1, 1)  # misura q1 -> c1

draw_mpl(qc_meas, title="Circuito con barriera e misure esplicite")
plt.show()

# Confronto stato puro vs distribuzione misurata
sv = Statevector.from_instruction(QuantumCircuit(2).h(0).cx(0, 1))
print("Probabilità teoriche (Born):")
for label, amp in zip(["00", "01", "10", "11"], sv.data):
    print(f"  |{label}⟩: {abs(amp)**2:.4f}")

counts = run_circuit(qc_meas, shots=4096)
print("\nConteggi simulati:", counts)

---
## 7. Superdense coding (Alice e Bob)

**Problema:** Alice vuole inviare **2 bit classici** a Bob usando **1 solo qubit** trasmesso.

**Risorsa condivisa:** una coppia EPR (stato Bell) preparata in anticipo:
$$|\Phi^+\rangle_{AB} = \frac{|00\rangle + |11\rangle}{\sqrt{2}}$$

**Protocollo:**
1. Alice e Bob condividono $|\Phi^+\rangle$: Alice ha il qubit $A$, Bob il qubit $B$.
2. Alice codifica i suoi 2 bit $(b_0, b_1)$ con una delle 4 operazioni di Pauli sul suo qubit:
   - $00 \to I$, $\; 01 \to X$, $\; 10 \to Z$, $\; 11 \to iY = ZX$
3. Alice **invia** il suo qubit a Bob (1 trasmissione quantistica).
4. Bob applica **CNOT** e **H** e misura entrambi i qubit, recuperando $(b_0, b_1)$.

**Intuizione:** l'entanglement è una risorsa che, insieme a 1 qubit trasmesso, permette di trasferire
2 bit — il doppio della capacità classica di un singolo qubit (che codifica al massimo 1 bit di
informazione classica accessibile).

In [ ]:
def superdense_coding_circuit(b0: int, b1: int) -> QuantumCircuit:
    """Circuito completo: EPR + codifica Alice + decodifica Bob."""
    qc = QuantumCircuit(2, 2)
    # Preparazione EPR (Alice = q0, Bob = q1)
    qc.h(0)
    qc.cx(0, 1)
    qc.barrier(label="EPR condiviso")
    # Codifica Alice su q0
    if b0:
        qc.x(0)
    if b1:
        qc.z(0)
    qc.barrier(label="Alice codifica")
    # Decodifica Bob
    qc.cx(0, 1)
    qc.h(0)
    qc.measure([0, 1], [0, 1])
    return qc


for bits, label in [((0, 0), "00"), ((0, 1), "01"), ((1, 0), "10"), ((1, 1), "11")]:
    qc_sd = superdense_coding_circuit(*bits)
    draw_mpl(qc_sd, title=f"Superdense coding — Alice invia {label}")
    plt.show()
    counts = run_circuit(qc_sd, shots=512, show_hist=False)
    dominant = max(counts, key=counts.get)
    print(f"Bit inviati {label} -> misura dominante: {dominant}")
    print(counts, "\n")

---
## 8. Teletrasporto quantistico

**Obiettivo:** trasferire uno **stato quantistico sconosciuto** $|\psi\rangle = \alpha|0\rangle+\beta|1\rangle$
da Alice a Bob **senza** inviare fisicamente il qubit, usando:
- 1 coppia EPR condivisa
- 2 bit classici (comunicazione standard)

**Protocollo (Bennett et al., 1993):**
1. Alice ha il qubit $|\psi\rangle$ (q0) e metà EPR (q1); Bob ha l'altra metà (q2).
2. Alice esegue **CNOT(q0, q1)** e **H(q0)**, poi misura q0 e q1 → 2 bit classici $(m_0, m_1)$.
3. Bob riceve $(m_0, m_1)$ e applica correzioni sul suo qubit:
   - se $m_1=1$: $X$ su q2
   - se $m_0=1$: $Z$ su q2
4. Il qubit di Bob è ora in $|\psi\rangle$.

**Nota:** il teletrasporto **non viola** il no-cloning: lo stato originale di Alice viene distrutto
dalla misura; si trasferisce, non si copia.

In [ ]:
def teleportation_circuit(alpha: complex, beta: complex) -> QuantumCircuit:
    qc = QuantumCircuit(3, 2)
    # Qubit da teletrasportare (q0)
    qc.initialize([alpha, beta], 0)
    # Coppia EPR Alice(q1)-Bob(q2)
    qc.h(1)
    qc.cx(1, 2)
    qc.barrier(label="risorsa EPR")
    # Misura di Bell lato Alice
    qc.cx(0, 1)
    qc.h(0)
    qc.measure(0, 0)
    qc.measure(1, 1)
    # Correzioni condizionali lato Bob (Qiskit 2.x: if_test)
    with qc.if_test((1, 1)):
        qc.x(2)
    with qc.if_test((0, 1)):
        qc.z(2)
    return qc


# Stato arbitrario da teletrasportare
theta, phi = np.pi / 3, np.pi / 4
alpha = np.cos(theta / 2)
beta = np.exp(1j * phi) * np.sin(theta / 2)

qc_tele = teleportation_circuit(alpha, beta)
draw_mpl(qc_tele, title="Teletrasporto quantistico")
plt.show()

# Verifica: stato su q2 dopo teletrasporto (senza misura finale su q2)
qc_check = QuantumCircuit(3)
qc_check.initialize([alpha, beta], 0)
qc_check.h(1)
qc_check.cx(1, 2)
qc_check.cx(0, 1)
qc_check.h(0)
# Simuliamo tutte le combinazioni di correzioni
target = Statevector([alpha, beta])
print("Stato target |ψ⟩:", np.round(target.data, 4))

for m0 in [0, 1]:
    for m1 in [0, 1]:
        qc_branch = QuantumCircuit(3)
        qc_branch.initialize([alpha, beta], 0)
        qc_branch.h(1)
        qc_branch.cx(1, 2)
        qc_branch.cx(0, 1)
        qc_branch.h(0)
        if m1:
            qc_branch.x(2)
        if m0:
            qc_branch.z(2)
        sv = Statevector.from_instruction(qc_branch)
        # Proietta su q2 (traccia su q0, q1)
        rho2 = sv.partial_trace([0, 1])
        fidelity = abs(np.vdot(target.data, rho2.data[:, 0])) ** 2
        print(f"  m0={m0}, m1={m1} -> fedeltà con |ψ⟩: {fidelity:.4f}")

---
## 9. Algoritmo di Bernstein–Vazirani

**Problema:** data una funzione nascosta $f:\{0,1\}^n \to \{0,1\}$ del tipo
$$f(\mathbf{x}) = \mathbf{s} \cdot \mathbf{x} \pmod{2}$$
dove $\mathbf{s}$ è una stringa binaria segreta, trovare $\mathbf{s}$.

**Classico:** servono fino a $n$ query (provare ogni bit di $\mathbf{s}$).
**Quantistico:** **1 sola query** grazie all'interferenza.

**Circuito:**
1. Ancilla $| - \rangle = H|1\rangle$ sul qubit ausiliario
2. $H^{\otimes n}$ sui qubit di input → sovrapposizione uniforme su tutte le $\mathbf{x}$
3. Oracolo $U_f:|\mathbf{x}\rangle|y\rangle \mapsto |\mathbf{x}\rangle|y \oplus f(\mathbf{x})\rangle$
   implementato con CNOT controllati dalla stringa $\mathbf{s}$
4. $H^{\otimes n}$ di nuovo → interferenza costruttiva su $\mathbf{s}$
5. Misura → si legge direttamente $\mathbf{s}$

In [ ]:
def bernstein_vazirani(secret: str) -> QuantumCircuit:
    n = len(secret)
    qc = QuantumCircuit(n + 1, n)
    qc.x(n)          # ancilla |1⟩
    qc.h(n)          # ancilla |−⟩
    qc.h(range(n))   # sovrapposizione uniforme
    # Oracolo: CNOT dal qubit i al target se secret[i]=='1'
    for i, bit in enumerate(reversed(secret)):
        if bit == "1":
            qc.cx(i, n)
    qc.h(range(n))
    qc.measure(range(n), range(n))
    return qc


secret = "1011"
qc_bv = bernstein_vazirani(secret)
draw_mpl(qc_bv, title=f"Bernstein–Vazirani — stringa segreta s = {secret}")
plt.show()

counts = run_circuit(qc_bv, shots=1024)
measured = max(counts, key=counts.get)
print(f"Stringa segreta: {secret}")
print(f"Misura dominante: {measured} (atteso: {secret[::-1]} in convenzione qiskit little-endian)")

---
## 10. Algoritmo di Deutsch

**Problema:** determinare se una funzione $f:\{0,1\}\to\{0,1\}$ è:
- **costante** ($f(0)=f(1)$), oppure
- **bilanciata** ($f(0)\neq f(1)$)

**Classico:** 2 valutazioni di $f$ nel caso peggiore.
**Quantistico (Deutsch):** **1 query** con interferenza.

**Idea:** preparare l'ancilla in $|-\rangle$, sovrapporre gli input con $H$, applicare l'oracolo,
e applicare $H$ sull'input: se $f$ è costante si misura $|0\rangle$, se bilanciata si misura $|1\rangle$.

Testiamo le 4 funzioni possibili: $f(x)=0$, $f(x)=1$, $f(x)=x$, $f(x)=1-x$.

In [ ]:
def deutsch_oracle(f_type: str) -> QuantumCircuit:
    """Oracolo a 2 qubit per le 4 funzioni 1-bit."""
    qc = QuantumCircuit(2)
    if f_type == "f0":      # f(x)=0
        pass
    elif f_type == "f1":    # f(x)=1
        qc.x(1)
    elif f_type == "id":    # f(x)=x
        qc.cx(0, 1)
    elif f_type == "not":   # f(x)=1-x
        qc.x(0)
        qc.cx(0, 1)
        qc.x(0)
    else:
        raise ValueError(f_type)
    return qc


def deutsch_circuit(f_type: str) -> QuantumCircuit:
    qc = QuantumCircuit(2, 1)
    qc.x(1)
    qc.h(1)
    qc.h(0)
    qc.compose(deutsch_oracle(f_type), inplace=True)
    qc.h(0)
    qc.measure(0, 0)
    return qc


for f_type, desc, expected in [
    ("f0", "f(x)=0 (costante)", "0"),
    ("f1", "f(x)=1 (costante)", "0"),
    ("id", "f(x)=x (bilanciata)", "1"),
    ("not", "f(x)=1-x (bilanciata)", "1"),
]:
    qc_d = deutsch_circuit(f_type)
    draw_mpl(qc_d, title=f"Deutsch — {desc}")
    plt.show()
    counts = run_circuit(qc_d, shots=512, show_hist=False)
    print(f"{desc}: {counts} (atteso |{expected}⟩ dominante)\n")

---
## 11. Algoritmo di Grover

**Problema:** cercare un elemento marcato in un database non strutturato di $N = 2^n$ elementi.

**Classico:** $O(N)$ query nel caso medio.
**Grover (1996):** $O(\sqrt{N})$ query — **speedup quadratico**.

**Schema:**
1. **Sovrapposizione uniforme** con $H^{\otimes n}$
2. **Oracolo** $O$: inverte la fase dello stato marcato $|w\rangle$: $O|w\rangle = -|w\rangle$
3. **Diffusore** $D = 2|\psi\rangle\langle\psi| - I$: riflette rispetto alla media
4. Ripetere $O(\sqrt{N})$ volte

**Demo:** 2 qubit ($N=4$), stato marcato $|11\rangle$. Con 1 iterazione di Grover la probabilità
di misurare $|11\rangle$ sale da $1/4$ a $1$.

In [ ]:
def grover_oracle_2q(marked: str) -> QuantumCircuit:
    """Oracolo che flippa la fase di |marked⟩ (2 qubit)."""
    qc = QuantumCircuit(2)
    # Porta |marked⟩ -> |11⟩ con X sui qubit dove marked[i]=0, poi Z controllata, poi undo
    for i, bit in enumerate(reversed(marked)):
        if bit == "0":
            qc.x(i)
    qc.cz(0, 1)
    for i, bit in enumerate(reversed(marked)):
        if bit == "0":
            qc.x(i)
    return qc


def grover_diffuser_2q() -> QuantumCircuit:
    qc = QuantumCircuit(2)
    qc.h([0, 1])
    qc.x([0, 1])
    qc.cz(0, 1)
    qc.x([0, 1])
    qc.h([0, 1])
    return qc


def grover_2q(marked: str = "11", iterations: int = 1) -> QuantumCircuit:
    qc = QuantumCircuit(2, 2)
    qc.h([0, 1])
    for _ in range(iterations):
        qc.compose(grover_oracle_2q(marked), inplace=True)
        qc.compose(grover_diffuser_2q(), inplace=True)
    qc.measure([0, 1], [0, 1])
    return qc


marked = "11"
qc_grover = grover_2q(marked, iterations=1)
draw_mpl(qc_grover, title=f"Grover 2-qubit — cerca |{marked}⟩")
plt.show()

counts = run_circuit(qc_grover, shots=2048)
print(f"Probabilità |{marked}⟩: {counts.get(marked, 0) / 2048:.2%}")

---
## 12. Algoritmo di Shor (demo educativa)

**Problema:** fattorizzare un intero $N$ in tempo polinomiale — impossibile in modo efficiente
con algoritmi classici noti (base della crittografia RSA).

**Idea di Shor (1994):** ridurre la fattorizzazione al problema di trovare il **periodo** $r$ della
funzione $f(x) = a^x \bmod N$ per un $a$ scelto a caso. Se $r$ è pari e $a^{r/2} \not\equiv -1 \pmod N$,
allora $\gcd(a^{r/2}\pm 1, N)$ dà un fattore non banale.

**Componenti quantistici:**
1. **QFT** (Quantum Fourier Transform) per estrarre il periodo dalla sovrapposizione
2. **Modular exponentiation** controllata

Qiskit include un circuito predefinito per la fattorizzazione. Usiamo la demo classica su $N=15$, $a=7$
(periodo $r=4$, fattori 3 e 5) e visualizziamo il circuito Shor semplificato.

In [ ]:
from math import gcd
from fractions import Fraction
from qiskit.circuit.library import QFT


def c_amod15(a: int, power: int):
    """Porta controllata per moltiplicazione mod 15 (demo educativa, N=15)."""
    if a not in [2, 7, 8, 11, 13]:
        raise ValueError("'a' deve essere uno tra [2, 7, 8, 11, 13]")
    U = QuantumCircuit(4)
    for _ in range(power):
        if a in [2, 13]:
            U.swap(0, 1); U.swap(1, 2); U.swap(2, 3)
        if a in [7, 8]:
            U.swap(2, 3); U.swap(1, 2); U.swap(0, 1)
        if a == 11:
            U.swap(1, 3); U.swap(0, 2)
        if a in [7, 11, 13]:
            for q in range(4):
                U.x(q)
    gate = U.to_gate()
    gate.name = f"{a}^{power} mod 15"
    return gate.control()


def shor_period_circuit(a: int = 7, n_count: int = 8) -> QuantumCircuit:
    """Circuito di phase estimation per trovare il periodo di a^x mod 15."""
    qc = QuantumCircuit(n_count + 4, n_count)
    for q in range(n_count):
        qc.h(q)
    qc.x(3 + n_count)  # |1> nel registro di lavoro
    for q in range(n_count):
        qc.append(c_amod15(a, 2**q), [q] + [i + n_count for i in range(4)])
    qc.append(QFT(n_count, inverse=True), range(n_count))
    qc.measure(range(n_count), range(n_count))
    return qc


# --- Parte classica (idea di Shor) ---
N, a = 15, 7
print(f"Fattorizzazione di N={N} con a={a}")
print("Sequenza a^x mod N:", [pow(a, x, N) for x in range(12)])
r = 4  # periodo noto per (7, 15)
half = pow(a, r // 2, N)
f1, f2 = gcd(half - 1, N), gcd(half + 1, N)
print(f"Periodo r={r}, a^(r/2) mod N = {half}")
print(f"Fattori: gcd({half}-1, {N})={f1}, gcd({half}+1, {N})={f2}")

# --- Circuito quantistico (phase estimation + QFT inversa) ---
qc_shor = shor_period_circuit(a=7, n_count=8)
print(f"\nCircuito Shor educativo: {qc_shor.num_qubits} qubit totali")
display(qc_shor.decompose().draw(output="text", fold=80))

counts = run_circuit(qc_shor, shots=1024, show_hist=True)
measured = max(counts, key=counts.get)
phase = int(measured, 2) / (2**8)
frac = Fraction(phase).limit_denominator(N)
r_est = frac.denominator if frac.numerator else 1
print(f"Misura dominante: {measured} -> fase ~ {phase:.4f} -> periodo stimato r ~ {r_est}")
if r_est % 2 == 0:
    half_est = pow(a, r_est // 2, N)
    print(f"Fattori stimati: gcd({half_est}-1, {N})={gcd(half_est-1, N)}, gcd({half_est}+1, {N})={gcd(half_est+1, N)}")

---
## 13. (Opzionale) Esecuzione su hardware IBM

Per inviare circuiti a un processore quantistico reale:
1. Crea un account su [IBM Quantum](https://quantum.ibm.com/)
2. Copia `qiskit_secrets.example.py` → `qiskit_secrets.local.py` e inserisci il token
3. `pip install qiskit-ibm-runtime`
4. Esegui la cella sotto

In [ ]:
ibm_backend = try_ibm_backend()
if ibm_backend is not None:
    qc_ibm = QuantumCircuit(1, 1)
    qc_ibm.h(0)
    qc_ibm.measure(0, 0)
    tqc = transpile(qc_ibm, ibm_backend)
    print(f"Circuito traspilato per {ibm_backend.name}: {tqc.num_qubits} qubit fisici")
    display(tqc.draw(output="mpl"))
    # Per eseguire davvero: from qiskit_ibm_runtime import SamplerV2 ...
    print("(Esecuzione reale commentata — decommentare con token valido)")

---
## Riepilogo

| Argomento | Concetto chiave |
|-----------|----------------|
| Pauli X,Y,Z | Rotazioni π; bit-flip e phase-flip |
| Hadamard | Sovrapposizione e interferenza |
| CNOT | Entanglement, logica controllata |
| Z | Fase; visibile solo con interferenza |
| Misura | Collasso stocastico (regola di Born) |
| Superdense coding | 2 bit con 1 qubit + EPR |
| Teletrasporto | Trasferimento stato via EPR + 2 bit classici |
| Bernstein–Vazirani | Stringa nascosta in 1 query |
| Deutsch | Costante vs bilanciata in 1 query |
| Grover | Ricerca in O(√N) |
| Shor | Fattorizzazione via periodo + QFT |

**Prossimi passi suggeriti:** variare i parametri (stringhe BV, stati Grover, altri $N$ per Shor),
confrontare fedeltà con rumore reale su IBM, collegare le matrici di `quantum/_common/qutils.py`
ai circuiti Qiskit.